In [10]:
# autoreload
%load_ext autoreload
%autoreload 2
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
import gc
import numpy as np
import gradio as gr
import json 
import re
import subprocess
import IPython.display as ipd
import torch
import torchaudio

from einops import rearrange
from safetensors.torch import load_file
from torch.nn import functional as F
from torchaudio import transforms as T

from stable_audio_tools.interface.aeiou import audio_spectrogram_image
from stable_audio_tools.inference.generation import generate_diffusion_cond, generate_diffusion_cond_inpaint, generate_diffusion_uncond
from stable_audio_tools.models.factory import create_model_from_config
from stable_audio_tools.models.pretrained import get_pretrained_model
from stable_audio_tools.models.utils import copy_state_dict, load_ckpt_state_dict
from stable_audio_tools.inference.utils import prepare_audio
from stable_audio_tools.loraw.network import LoRAMerger, create_lora_from_config

from stable_audio_tools.interface.interfaces.diffusion_cond import create_diffusion_cond_ui

model = None
model_type = None
sample_rate = 32000
sample_size = 1920000

def load_model(model_config=None, model_ckpt_path=None, pretrained_name=None, pretransform_ckpt_path=None, device="cuda", model_half=False, use_lora=True):
    global model, sample_rate, sample_size
    
    if pretrained_name is not None:
        print(f"Loading pretrained model {pretrained_name}")
        model, model_config = get_pretrained_model(pretrained_name)

    elif model_config is not None and model_ckpt_path is not None:
        print(f"Creating model from config")
        model = create_model_from_config(model_config)

        print(f"Loading model checkpoint from {model_ckpt_path}")
        # Load checkpoint
        copy_state_dict(model, load_ckpt_state_dict(model_ckpt_path))
        #model.load_state_dict(load_ckpt_state_dict(model_ckpt_path))

    sample_rate = model_config["sample_rate"]
    sample_size = model_config["sample_size"]

    if pretransform_ckpt_path is not None:
        print(f"Loading pretransform checkpoint from {pretransform_ckpt_path}")
        model.pretransform.load_state_dict(load_ckpt_state_dict(pretransform_ckpt_path), strict=False)
        print(f"Done loading pretransform")

    model.to(device).eval().requires_grad_(False)

    if model_half:
        model.to(torch.float16)
    if use_lora:
        lora = create_lora_from_config(model_config, model)
        return lora, model, model_config
    else:
        return None, model, model_config


model_config_path="/home/zachary/code/stable-audio-tools/stable_audio_tools/configs/model_configs/txt2audio/sao_short_inpaint_silence_encenc_postpend.json"
ckpt_path="/home/zachary/.cache/huggingface/hub/models--stabilityai--stable-audio-open-1.0/snapshots/f21265c1e2710b3bd2386596943f0007f55f802e/model.safetensors"
# model_config_path = "/home/zachary/code/stable-audio-tools/stable_audio_tools/configs/model_configs/txt2audio/saos_inpaint_silence_encenc_postpend.json"
# ckpt_path = "/home/zachary/code/stable-audio-tools/saos-encenc.safetensors"

if model_config_path is not None:
        # Load config from json file
    with open(model_config_path) as f:
        model_config = json.load(f)
else:
    model_config = None

device = "cuda" if torch.cuda.is_available() else "cpu"

lora, sao, conf = load_model(model_config, ckpt_path,  device=device, model_half=True, use_lora=True)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Creating model from config


/home/zachary/miniconda3/envs/sat/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Loading model checkpoint from /home/zachary/.cache/huggingface/hub/models--stabilityai--stable-audio-open-1.0/snapshots/f21265c1e2710b3bd2386596943f0007f55f802e/model.safetensors
Full-finetune list: ['to_input_add_embed']
Found 175 candidates for LoRA replacement
LoRA module added: model/model/to_timestep_embed/0
LoRA module added: model/model/to_timestep_embed/2
LoRA module added: model/model/to_cond_embed/0
LoRA module added: model/model/to_cond_embed/2
Full-finetune module added: model/model/to_input_add_embed
LoRA module added: model/model/transformer/layers/0/self_attn/to_qkv
LoRA module added: model/model/transformer/layers/0/self_attn/to_out
LoRA module added: model/model/transformer/layers/0/cross_attn/to_q
LoRA module added: model/model/transformer/layers/0/cross_attn/to_kv
LoRA module added: model/model/transformer/layers/0/cross_attn/to_out
LoRA module added: model/model/transformer/layers/0/ff/ff/0/proj
LoRA module added: model/model/transformer/layers/0/ff/ff/2
LoRA module

In [11]:
print(lora.net.lora_modules['model/model/to_timestep_embed/0'].lora_down.weight.data)
print(lora.net.full_ft_modules['model/model/to_input_add_embed'].weight.data)
print(sao.model.model.to_input_add_embed.weight.data)

tensor([[-0.0207,  0.0562,  0.0059,  ..., -0.0220,  0.0394, -0.0293],
        [ 0.0010,  0.0207, -0.0622,  ...,  0.0487,  0.0003, -0.0382],
        [-0.0215, -0.0354, -0.0135,  ..., -0.0162, -0.0002,  0.0357],
        ...,
        [-0.0043, -0.0557,  0.0071,  ..., -0.0497, -0.0447, -0.0227],
        [ 0.0556,  0.0227, -0.0417,  ..., -0.0578,  0.0108, -0.0154],
        [-0.0139,  0.0612,  0.0050,  ...,  0.0384, -0.0030,  0.0320]])
tensor([[-0.1168,  0.0015,  0.0955,  ..., -0.0967,  0.0610, -0.1117],
        [ 0.0338, -0.0414,  0.0950,  ..., -0.0228,  0.0316, -0.0391],
        [ 0.0971,  0.0226,  0.1104,  ..., -0.0309,  0.0229,  0.0574],
        ...,
        [-0.0708, -0.0851, -0.0391,  ...,  0.0295,  0.0662, -0.0211],
        [ 0.0782,  0.0106,  0.0844,  ..., -0.0133,  0.0726,  0.1008],
        [ 0.0936, -0.1089, -0.1199,  ..., -0.0606,  0.0827,  0.0065]],
       device='cuda:0', dtype=torch.float16)
tensor([[-0.1168,  0.0015,  0.0955,  ..., -0.0967,  0.0610, -0.1117],
        [ 0.0338,

In [12]:
import torch

lora_state_d = torch.load("/home/zachary/checkpoints/s2s/ossl2_experiments/akm2d6mj/checkpoints/epoch=56-step=125000.ckpt", map_location="cpu")
lora.load_weights(lora_state_d)
lora.activate()

# lora.net.to(torch.float16)

Injected 174 LoRA modules into model


In [13]:
print(lora.net.lora_modules['model/model/to_timestep_embed/0'].lora_down.weight.data)
print(lora.net.full_ft_modules['model/model/to_input_add_embed'].weight.data)
print(sao.model.model.to_input_add_embed.weight.data)

tensor([[ 0.0510,  0.0298,  0.0396,  ...,  0.0057,  0.0133,  0.0358],
        [ 0.0402,  0.0685, -0.0030,  ..., -0.0479,  0.0361,  0.0440],
        [ 0.0700,  0.0364, -0.0726,  ...,  0.0229,  0.0317,  0.0342],
        ...,
        [-0.0297, -0.0133,  0.0340,  ...,  0.0027, -0.0040, -0.0366],
        [-0.0243, -0.0143, -0.0212,  ..., -0.0761,  0.0418, -0.0248],
        [ 0.0448,  0.0527,  0.0078,  ..., -0.0035, -0.0400, -0.0669]])
tensor([[-0.0355,  0.0095,  0.0198,  ...,  0.0387,  0.1171,  0.1220],
        [ 0.0240,  0.0316,  0.1026,  ..., -0.0086,  0.0191, -0.0143],
        [ 0.0264,  0.0859, -0.0332,  ...,  0.0306, -0.0142,  0.0358],
        ...,
        [-0.0455,  0.0201,  0.0291,  ..., -0.1346,  0.0295,  0.0458],
        [ 0.0638,  0.0944, -0.0196,  ...,  0.0077, -0.1344, -0.0681],
        [-0.0733, -0.1204,  0.0209,  ..., -0.1087,  0.0565, -0.0129]],
       device='cuda:0', dtype=torch.float16)
tensor([[-0.0355,  0.0095,  0.0198,  ...,  0.0387,  0.1171,  0.1220],
        [ 0.0240,

In [2]:
ref_audio, sr = torchaudio.load("/home/zachary/code/stable-audio-tools/notebooks/demo_cfg_4_320962_9da78d7d47814a3c52fa.wav")
ref_audio = ref_audio[:, 524288:524288*2]
# play reference audio
# ipd.display(ipd.Audio(ref_audio.cpu().numpy(), rate=sr))

In [3]:
# encode reference audio
ref_audio_prepared = prepare_audio(ref_audio, sr, sample_rate, sample_size, 2, device=device)
with torch.no_grad():
    ref_latents = sao.pretransform.encode(ref_audio_prepared.to(next(sao.pretransform.parameters()).dtype))


# decode to check
# with torch.no_grad():
#     recon_audio = sao.pretransform.decode(ref_latents)
# ipd.display(ipd.Audio(recon_audio[0].cpu().numpy(), rate=sample_rate))

In [4]:
sao = sao.to(device)
sao.model.model = torch.compile(sao.model.model)

In [5]:
from stable_audio_tools.inference.generation import generate_diffusion_cond_blockar
from stable_audio_tools.models.inpainting import random_inpaint_mask
os.environ["USE_CHECKPOINTING"] = "0"

# torch.backends.cudnn.enabled = False

sample_rate = model_config["sample_rate"]
sample_size = model_config["sample_size"]
enc_enc = model_config["training"].get("enc_enc", False)
ee_attn_pattern = model_config['training']['inpainting']['mask_kwargs'].get('enc_enc_attention_pattern', 'enc-dec')




# Set up text and timing conditioning
conditioning = [{
    "prompt": "This song is a relaxing electronic track with a steady beat around 130 BPM and vibrant synth lines, perfect for studying. The music features exciting synth textures, a driving rhythm, and a sparkling arpeggio lead that creates a calming atmosphere.",
    # "prompt": "This is a mournful and introspective orchestral piece, featuring a slow tempo, minor key, and rich harmonies. The music evokes a sense of longing and melancholy, with sweeping string sections, haunting woodwinds, and a somber piano melody that tugs at the heartstrings.",
    "seconds_start": 0, 
    "seconds_total": 12
}]

# mask latents
# inpaint_masked_input, inpaint_mask = random_inpaint_mask(ref_latents, torch.ones_like(ref_latents), **model_config['training']['inpainting']['mask_kwargs'])


# conditioning_tensors = model.conditioner(conditioning, device)
# conditioning_tensors['inpaint_mask'] = [inpaint_mask]
# conditioning_tensors['inpaint_masked_input'] = [inpaint_masked_input]
with torch.no_grad():
    with torch.cuda.amp.autocast(enabled=True, dtype=torch.float16):
        output2 = generate_diffusion_cond_blockar(
            sao,
            steps=50,
            cfg_scale=7,
            conditioning=conditioning,
            sample_size=sample_size,
            init_audio=(sr, ref_audio), # turn this on if you want ~10 seconds of initial audio to condition on
            sigma_min=0,
            sigma_max=1,
            sampler_type="euler",
            device=device,
            ar_style='outpaint',
            block_size=96256, # 47 latents per block
            generation_length=962560,
            seed=789423,
            # silence_dir='begh',
            enc_enc=True,
            enc_enc_attention_pattern=ee_attn_pattern,
            postpend=True,
            use_kv_cache=True,
            prefill=True,
            speedtest=False
        )

output2 = rearrange(output2, "b d n -> d (b n)")
ipd.Audio(output2.cpu().numpy(), rate=sample_rate)

/tmp/ipykernel_3480087/222480596.py:31: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=True, dtype=torch.float16):


789423
Block size: 47, Sample size: 255
Using enc-enc attention with block size 47
Creating enc-enc self attention block mask with pattern enc-dec and fixed mask size 208 for sequence length 255
Created self attention block mask:
(0, 0)
████
████



In [8]:
from stable_audio_tools.inference.generation import generate_diffusion_cond_blockar
from stable_audio_tools.models.inpainting import random_inpaint_mask
os.environ["USE_CHECKPOINTING"] = "0"

# torch.backends.cudnn.enabled = False

sample_rate = model_config["sample_rate"]
sample_size = model_config["sample_size"]
enc_enc = model_config["training"].get("enc_enc", False)
ee_attn_pattern = model_config['training']['inpainting']['mask_kwargs'].get('enc_enc_attention_pattern', 'enc-dec')




# Set up text and timing conditioning
conditioning = [{
    "prompt": "This song is a relaxing electronic track with a steady beat around 130 BPM and vibrant synth lines, perfect for studying. The music features exciting synth textures, a driving rhythm, and a sparkling arpeggio lead that creates a calming atmosphere.",
    # "prompt": "This is a mournful and introspective orchestral piece, featuring a slow tempo, minor key, and rich harmonies. The music evokes a sense of longing and melancholy, with sweeping string sections, haunting woodwinds, and a somber piano melody that tugs at the heartstrings.",
    "seconds_start": 0, 
    "seconds_total": 12
}]

# mask latents
# inpaint_masked_input, inpaint_mask = random_inpaint_mask(ref_latents, torch.ones_like(ref_latents), **model_config['training']['inpainting']['mask_kwargs'])


# conditioning_tensors = model.conditioner(conditioning, device)
# conditioning_tensors['inpaint_mask'] = [inpaint_mask]
# conditioning_tensors['inpaint_masked_input'] = [inpaint_masked_input]
with torch.inference_mode():
    with torch.cuda.amp.autocast(enabled=True, dtype=torch.float16):
        output2 = generate_diffusion_cond_blockar(
            sao,
            steps=50,
            cfg_scale=7,
            conditioning=conditioning,
            sample_size=sample_size,
            init_audio=(sr, ref_audio), # turn this on if you want ~10 seconds of initial audio to condition on
            sigma_min=0,
            sigma_max=1,
            sampler_type="euler",
            device=device,
            ar_style='outpaint',
            block_size=96256, # 47 latents per block
            generation_length=962560,
            seed=789423,
            # silence_dir='begh',
            enc_enc=True,
            enc_enc_attention_pattern=ee_attn_pattern,
            postpend=True,
            use_kv_cache=True,
            prefill=True,
            speedtest=True
        )

output2 = rearrange(output2, "b d n -> d (b n)")
ipd.Audio(output2.cpu().numpy(), rate=sample_rate)

/tmp/ipykernel_3480087/2744407487.py:31: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=True, dtype=torch.float16):


789423
Block size: 47, Sample size: 255
Using enc-enc attention with block size 47
Creating enc-enc self attention block mask with pattern enc-dec and fixed mask size 208 for sequence length 255
Created self attention block mask:
(0, 0)
████
████

Average inference time per block: 152.508466796875 ms


RuntimeError: Tensor type unknown to einops <class 'NoneType'>

In [ ]:
# convert to spectrogram and look at it
# do this manually dawg using torchaudio.transforms.MelSpectrogram
mel_spec_transform = T.MelSpectrogram(
    sample_rate=sample_rate,
    n_fft=1024,
    hop_length=256,
    win_length=1024,
    f_min=0,
    f_max=sample_rate//2,
    pad=0,
    n_mels=80,
    power=2.0,
    normalized=False,
)
mel_spec = mel_spec_transform(torch.tensor(output).unsqueeze(0).cpu().mean(dim=1))
mel_spec_db = T.AmplitudeToDB(top_db=80)(mel_spec)
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 4))
plt.imshow(mel_spec_db.squeeze().cpu().numpy(), aspect='auto', origin='lower')
plt.colorbar(format='%+2.0f dB')
plt.title('Mel Spectrogram')
plt.xlabel('Time (s)')
# change x label to reflext time in seconds at 44.1kHz
plt.xticks(ticks=np.arange(0, mel_spec_db.shape[-1], step=mel_spec_db.shape[-1]//10), labels=[f"{(i * 256) / sample_rate:.1f}" for i in np.arange(0, mel_spec_db.shape[-1], step=mel_spec_db.shape[-1]//10)])
plt.ylabel('Mel Frequency')
plt.tight_layout()
plt.show()